# Load data

## Julia 
Need to be ran in terminal before start (for PubMedRAG)
### Install Julia
```
JULIA_VERSION=1.10.4
cd $HOME
wget https://julialang-s3.julialang.org/bin/linux/x64/1.10/julia-$JULIA_VERSION-linux-x86_64.tar.gz
tar -xvzf julia-$JULIA_VERSION-linux-x86_64.tar.gz
ln -s julia-$JULIA_VERSION julia

export PATH="$HOME/julia/bin:$PATH" # add to PATH (put this in ~/.bashrc or your sbatch script)
julia --version
```
#### Run Julia
```
!bash slurm/run_julia.sh
```

If you see the below message, you are good. (Takes ~2min)
```
Using device: cpu
<All keys matched successfully>
INFO:     Started server process [2509132]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8002 (Press CTRL+C to quit)
[ Info: Listening on: 0.0.0.0:8003, thread id: 14
```

In [20]:

import json
import os
from explain.util import load_data
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

from explain.llm.data_generator import DataGenerator
from explain.util import set_perturbation_ner_mapping
from explain.eval.score.evaluate import evaluate
from explain.starkprimekg.starkprimekg_utils import get_kg_info
from explain.starkprimekg.starkprimekg import StarkPrimeKG
from explain.literature.harmonizome_utils import get_harmonizome_info
from explain.literature.wikipedia_utils import get_wikipedia_info
from explain.literature.paperqa_utils import get_paperqa_info
# pubmed_info can be imported after running the above julia commands
from explain.literature.pubmed_utils import get_pubmed_info

In [4]:
os.chdir('/mnt/ps/home/CORP/yunhui.jang/research/hooke-explain')

In [6]:
# Need to set index_tag for dataset (Tahoe: tahoe_ , RXRX: rxrx_ , etc.)
index_tag = "tahoe_"
pert_path = '/rxrx/data/user/yunhui.jang/outgoing/tahoe/perturbation_ner_mapping_batch_0.json'
perturbation_ner_mapping = json.load(open(pert_path, 'r'))
perturbation_ner_mapping_data_indices = [index_tag + str(item['index']) for item in perturbation_ner_mapping]

In [7]:
set_perturbation_ner_mapping(perturbation_ner_mapping, perturbation_ner_mapping_data_indices)

In [22]:
report_list = []
structure_explain_list = []
# tool_list: select tools to use (["pubmed-fast-ner", "kg-ner", "harmonizome", "wikipedia"])
tool_list = ['pubmed-fast-ner', 'kg-ner', 'harmonizome', 'wikipedia']   
data_generator = DataGenerator(model_name='anthropic', tool_list=tool_list, pert_path=pert_path)
perturbations = data_generator.perturbation_cell_context


2025-11-27 09:28:37.767 | INFO     | explain.llm._client:__init__:694 - Initialized unified LLM client with provider: anthropic
2025-11-27 09:28:37.798 | INFO     | explain.llm._client:__init__:694 - Initialized unified LLM client with provider: anthropic


In [23]:
kg = StarkPrimeKG()

Loading embeddings from /rxrx/data/user/hamed.shirzad/outgoing/stark_prime_kg/pritamdeka/S-PubMedBERT-MS-MARCO/node_embeddings.pt


# Data generation

In [29]:
report_list = []
structure_explain_list = []

for i, perturbation in enumerate(tqdm(perturbations[:3])):
        index = index_tag + str(perturbation['index'])
        additional_info = ""
        kg_info = ""
        harmonizome_info = ""
        wikipedia_info = ""
        extracted_graph_info = {}
        papers_info = {}
        perturbation_text = json.dumps(perturbation, indent=4)
        question = "**Q: How does the following perturbation influence the cell in the described context, mechanistically and functionally?**\n\n"
        question += perturbation_text
        result = {'additional': ''}
        for tool_name in tool_list:
            # Load KG information
            if 'kg' in tool_name:
                kg_info, extracted_graph_info = get_kg_info(index, perturbation, tool_name, kg, False, 1)
                kg_info = data_generator.post_process_additional_info(kg_info, tool_name, perturbation)
                result["kg_info"] = kg_info
                result["extracted_graph_info"] = extracted_graph_info
                result['additional'] += kg_info
            # Load harmonizome information
            elif 'harmonizome' in tool_name:
                ner_flag = 'ner' in tool_name
                harmonizome_info = get_harmonizome_info(index, perturbation, is_ner=ner_flag)
                harmonizome_info = data_generator.post_process_additional_info(harmonizome_info, tool_name, perturbation)
                result["harmonizome_info"] = harmonizome_info
                result['additional'] += harmonizome_info
            # Load wikipedia information
            elif 'wikipedia' in tool_name:
                wikipedia_info = get_wikipedia_info(index, perturbation)
                wikipedia_info = data_generator.post_process_additional_info(wikipedia_info, tool_name, perturbation)
                result["wikipedia_info"] = wikipedia_info
                result['additional'] += wikipedia_info
            # Load pubmed information (only when julia is running)
            elif 'pubmed' in tool_name:
                pubmed_info = get_pubmed_info(index, perturbation, question, num_papers=10, mode=tool_name)
                pubmed_info = [json.dumps(info, indent=4) for info in pubmed_info]
                pubmed_info = '\n'.join(pubmed_info)
                pubmed_info = data_generator.post_process_additional_info(pubmed_info, tool_name, perturbation)
                result["pubmed_info"] = pubmed_info
                result['additional'] = pubmed_info
        # report generation
        additional_info += result['additional']
        report = data_generator.generate_report(perturbation, additional_info)
        report_dict = {'index': index, 'perturbation': perturbation, 'report_text': report, 'question': question,
                'kg_info': kg_info, 'extracted_graph_info': extracted_graph_info, 'harmonizome_info': harmonizome_info,
                'wikipedia_info': wikipedia_info, 'pubmed_info': pubmed_info}
        report_list.append(report_dict)
        # structure explain generation
        structure_explain = data_generator.generate_structure_explain(report, question)
        thinking, answer, explain, dag = data_generator.process_structure_explain(structure_explain)
        structure_explain_dict = {'index': index, 'input_perturbation': perturbation, 'thinking': thinking,
        'answer': answer, 'explain': explain, 'dag': dag, 'raw_response': structure_explain,
        'question': question, 'input_report_text': report, 'additional_info': additional_info}
        structure_explain_list.append(structure_explain_dict)

  0%|          | 0/3 [00:00<?, ?it/s]/mnt/ps/home/CORP/yunhui.jang/research/hooke-explain/src/explain/starkprimekg/starkprimekg_utils.py:125: UserWarning: Could not find the node index for CellLine NCI-H1573 in the KG.
  warnings.warn(f"Could not find the node index for {tag} {entity} in the KG.")
/mnt/ps/home/CORP/yunhui.jang/research/hooke-explain/src/explain/starkprimekg/starkprimekg_utils.py:125: UserWarning: Could not find the node index for Disease None in the KG.
  warnings.warn(f"Could not find the node index for {tag} {entity} in the KG.")
/mnt/ps/home/CORP/yunhui.jang/research/hooke-explain/src/explain/starkprimekg/starkprimekg_utils.py:125: UserWarning: Could not find the node index for Disease GoF in the KG.
  warnings.warn(f"Could not find the node index for {tag} {entity} in the KG.")
/mnt/ps/home/CORP/yunhui.jang/research/hooke-explain/src/explain/starkprimekg/starkprimekg_utils.py:125: UserWarning: Could not find the node index for Disease LoF in the KG.
  warnings.warn

# Data

In [30]:
import pandas as pd

report_df = pd.DataFrame(report_list)
structure_explain_df = pd.DataFrame(structure_explain_list)

In [31]:
# XX_info: retreived information from XX tool
# report_text: generated report
report_df

,index,perturbation,report_text,question,kg_info,extracted_graph_info,harmonizome_info,wikipedia_info,pubmed_info
0,tahoe_0,"{'index': 0, 'perturbation': {'context': [{'ce...",# Mechanistic Report: Bestatin Effects on NCI-...,**Q: How does the following perturbation influ...,## KNOWLEDGE GRAPH INFORMATION\n- name: FBXW7 ...,"{'node_list': ['6017', '12936', '1785', '291',...",## GENE INFORMATION\n('### CELL TYPE INFORMATI...,## WIKIPEDIA INFORMATION\n## KRAS\nKRAS (Kirst...,"## RELATED PAPER LIST\n{\n ""title"": ""Clinic..."
1,tahoe_1,"{'index': 1, 'perturbation': {'context': [{'ce...",# Mechanistic Report: Bestatin Effects on NCI-...,**Q: How does the following perturbation influ...,## KNOWLEDGE GRAPH INFORMATION\n- name: ARID1A...,"{'node_list': ['6019', '1831', '4584', '1641',...",## GENE INFORMATION\n('### CELL TYPE INFORMATI...,## WIKIPEDIA INFORMATION\n## LoF\nLof (Spanish...,"## RELATED PAPER LIST\n{\n ""title"": ""Chemic..."
2,tahoe_2,"{'index': 2, 'perturbation': {'context': [{'ce...",# Mechanistic Report: Bestatin Effects on hTER...,**Q: How does the following perturbation influ...,## KNOWLEDGE GRAPH INFORMATION\n- name: TERT -...,"{'node_list': ['5129', '18990']}","## GENE INFORMATION\n('', {'gene_info': [], 'g...",## WIKIPEDIA INFORMATION\n## Bestatin\nUbenime...,"## RELATED PAPER LIST\n{\n ""title"": ""A revi..."


In [32]:
print(report_df.iloc[0]['kg_info'])

## KNOWLEDGE GRAPH INFORMATION
- name: FBXW7 - type: gene/protein - source: NCBI - details:   - query: FBXW7   - alias (other gene names): ['AGO', 'CDC4', 'DEDHIL', 'FBW6', 'FBW7', 'FBX30', 'FBXO30', 'FBXW6', 'SEL-10', 'SEL10', 'hAgo', 'hCdc4']   - genomic_pos (genomic position): {'chr': '4', 'end': 152536092, 'ensemblgene': 'ENSG00000109670', 'start': 152320544, 'strand': -1}   - name (gene name): F-box and WD repeat domain containing 7   - summary (protein summary text): This gene encodes a member of the F-box protein family which is characterized by an approximately 40 amino acid motif, the F-box. The F-box proteins constitute one of the four subunits of ubiquitin protein ligase complex called SCFs (SKP1-cullin-F-box), which function in phosphorylation-dependent ubiquitination. The F-box proteins are divided into 3 classes: Fbws containing WD-40 domains, Fbls containing leucine-rich repeats, and Fbxs containing either different protein-protein interaction modules or no recognizable 

In [33]:
print(report_df.iloc[0]['report_text'])

# Mechanistic Report: Bestatin Effects on NCI-H1573 Lung Cancer Cells

## 1. Perturbation Description

**Bestatin (Ubenimex) hydrochloride** is a competitive, reversible protease inhibitor derived from *Streptomyces olivoreticuli*. The compound is a dipeptide analog with the molecular formula corresponding to the SMILES string provided (CC(C)CC(C(=O)O)NC(=O)C(C(CC1=CC=CC=C1)N)O.Cl).

### Primary Targets and Mechanism:
- **Primary targets**: Aminopeptidase N (APN/CD13), aminopeptidase B, leukotriene A4 hydrolase, leucyl/cystinyl aminopeptidase, and membrane dipeptidase
- **Binding mechanism**: Competitive inhibition through binding to the active site of these zinc metallopeptidases
- **Molecular weight**: 308.38 Da
- **Mechanism of action**: Zinc metalloprotease inhibition, blocking enzymatic degradation of various peptides including oxytocin, vasopressin, and enkephalins

## 2. Cellular Context: NCI-H1573 Mutational Landscape

The NCI-H1573 non-small cell lung carcinoma cell line harbo

In [34]:
# explain: structured explanation
# answer: unstructured explanation (natural language)
# dag: DAG of the explanation
structure_explain_df

,index,input_perturbation,thinking,answer,explain,dag,raw_response,question,input_report_text,additional_info
0,tahoe_0,"{'index': 0, 'perturbation': {'context': [{'ce...",Let me work through this step-by-step:\n\n1. B...,Bestatin competitively binds and inhibits amin...,"set_context(cell_type=""NCI-H1573"", genotype=""K...","edge(""n1"", ""n2"", relation=""causal"")\nedge(""n1""...",<think>\nLet me work through this step-by-step...,**Q: How does the following perturbation influ...,# Mechanistic Report: Bestatin Effects on NCI-...,"## RELATED PAPER LIST\n{\n ""title"": ""Clinic..."
1,tahoe_1,"{'index': 1, 'perturbation': {'context': [{'ce...","Looking at the mechanistic report and context,...",Bestatin acts as a competitive inhibitor of mu...,"set_context(cell_type=""NCI-H460"", genotype=""KR...","edge(""n1"", ""n4"", relation=""causal"")\nedge(""n2""...",<think>\nLooking at the mechanistic report and...,**Q: How does the following perturbation influ...,# Mechanistic Report: Bestatin Effects on NCI-...,"## RELATED PAPER LIST\n{\n ""title"": ""Chemic..."
2,tahoe_2,"{'index': 2, 'perturbation': {'context': [{'ce...","Let me work through this step by step, focusin...",Bestatin competitively inhibits multiple amino...,"set_context(cell_type=""hTERT-HPNE pancreatic d...","edge(""n1"", ""n4"", relation=""causal"")\nedge(""n2""...",<think>\nLet me work through this step by step...,**Q: How does the following perturbation influ...,# Mechanistic Report: Bestatin Effects on hTER...,"## RELATED PAPER LIST\n{\n ""title"": ""A revi..."


In [35]:
# structured explain
print(structure_explain_df.iloc[0]['explain'])

set_context(cell_type="NCI-H1573", genotype="KRAS GoF, EGFR GoF, ERBB3 GoF, MET GoF, NRAS GoF, TP53 LoF, STK11 LoF, SMARCA4 LoF, PIK3R1 LoF, FBXW7 LoF, CDKN2A suppression", disease="non-small cell lung carcinoma", prior_perturbation="none")
  binds_to(id="n1", actor="Bestatin", target="CD13", affinity="competitive inhibition", via="zinc metallopeptidase active site")
  modulates_molecule_activity(id="n2", target="CD13", direction="down", via="competitive aminopeptidase inhibition")
  modulates_molecule_activity(id="n3", target="leukotriene A4 hydrolase", direction="down", via="metallopeptidase inhibition")
  modulates_pathway_activity(id="n4", pathway="peptide degradation", direction="down", via="aminopeptidase inhibition")
  modulates_pathway_activity(id="n5", pathway="leukotriene metabolism", direction="down", via="leukotriene A4 hydrolase inhibition")
  modulates_pathway_activity(id="n6", pathway="AMPK signaling", direction="down", via="metabolic stress interaction with STK11 LoF ba

In [36]:
# paragraph
print(structure_explain_df.iloc[0]['answer'])

Bestatin competitively binds and inhibits aminopeptidase N (CD13) and other zinc metallopeptidases with high specificity, blocking peptide degradation pathways. This causal inhibition leads to accumulation of bioactive peptides (oxytocin, vasopressin, enkephalins) and disruption of leukotriene metabolism, inducing metabolic stress. In the NCI-H1573 context, the pre-existing STK11 loss-of-function creates a synthetic vulnerability, as cells lack proper LKB1-AMPK energy sensing to cope with metabolic perturbations. The TP53 loss-of-function further impairs stress response coordination, while multiple oncogenic drivers (KRAS, EGFR, ERBB3) provide competing survival signals. This creates a correlative interaction where metabolic stress from aminopeptidase inhibition overcomes some oncogenic signaling, resulting in growth inhibition, metabolic stress gene expression, and potential selective cytotoxicity despite the hyperactivated growth pathways.
